# B9.16 hosted smoke test — Qwen3-TTS CustomVoice

This notebook performs **one** hosted inference attempt for Qwen3-TTS CustomVoice, using the exact canonical segment-0 text. It runs only when you execute the cells in Google Colab. It does not connect to or modify the production app, production narration, databases, or YouTube.

## Before running

1. Open this notebook in Google Colab.
2. In **Runtime → Change runtime type**, choose a GPU accelerator.
3. Select the available **past runtime version `2026.07` (Python 3.12.13)** if Colab offers it. Colab's backend-info repository documents that runtime as Python 3.12.13. If that runtime is not selectable, do not silently proceed with Python 3.13; capture the preflight failure and report it.
4. Run the single execution cell below. This notebook records installation and inference failures and exports a ZIP containing the logs and manifest. It does not claim PASS just because the cell completed.

Official source: https://github.com/QwenLM/Qwen3-TTS

**Hosted-only policy:** inference must run on the Colab runtime with a GPU. Do not run this notebook's model inference on the local laptop.

In [ ]:
# Canonical input and immutable expected hash
SEGMENT_0_TEXT = "How a single misspelling created the most iconic domain name in tech is about to change everything, and here's why..."
EXPECTED_TEXT_SHA256 = '7b292603c4308ecf92cc882e1e2cc21b5da57ccade71b51ec3078a57bb11f5a8'

# Model-specific settings
MODEL_NAME = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
RUN_LABEL = 'b9_16_qwen'

In [ ]:
# Run one complete hosted attempt, preserving diagnostics even on failure.
import os, sys, json, time, hashlib, traceback, subprocess, platform, zipfile
from pathlib import Path

RUN_DIR = Path('/content') / RUN_LABEL
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = RUN_DIR / 'execution.log'
MANIFEST_PATH = RUN_DIR / 'manifest.json'
AUDIO_PATH = RUN_DIR / ("qwen3_tts_customvoice.wav")
ZIP_PATH = Path('/content') / f'{RUN_LABEL}_artifacts.zip'

manifest = {
    'run_label': RUN_LABEL,
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'engine': 'Qwen3-TTS CustomVoice',
    'model_name': MODEL_NAME,
    'text_sha256_expected': EXPECTED_TEXT_SHA256,
    'text_sha256_actual': None,
    'text_length': len(SEGMENT_0_TEXT),
    'runtime': {'python': sys.version, 'platform': platform.platform()},
    'gpu': None,
    'package_install_status': 'not_started',
    'model_initialization_status': 'not_started',
    'inference_status': 'not_started',
    'technical_validation_status': 'not_started',
    'audio_quality_status': 'PENDING_HUMAN_LISTENING',
    'audio_path': str(AUDIO_PATH),
    'audio_sha256': None,
    'error': None,
}
log_parts = []
def log(msg):
    s = str(msg)
    print(s)
    log_parts.append(s)
def run_command(args, label):
    log(f"\n===== COMMAND: {label} =====\n$ {' '.join(args)}")
    p = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log(p.stdout)
    log(f"[exit_code={p.returncode}]")
    if p.returncode != 0:
        raise RuntimeError(f"Command failed (exit={p.returncode}): {label}; see execution.log")
    return p.stdout

def write_diagnostics():
    LOG_PATH.write_text('\n'.join(log_parts) + '\n', encoding='utf-8')
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
        if AUDIO_PATH.exists(): z.write(AUDIO_PATH, AUDIO_PATH.name)
        z.write(MANIFEST_PATH, MANIFEST_PATH.name)
        z.write(LOG_PATH, LOG_PATH.name)
    log(f"Artifacts: {ZIP_PATH}")

try:
    # Verify text identity before touching the model.
    actual_text_hash = hashlib.sha256(SEGMENT_0_TEXT.encode('utf-8')).hexdigest()
    manifest['text_sha256_actual'] = actual_text_hash
    if actual_text_hash != EXPECTED_TEXT_SHA256:
        raise RuntimeError(f'Text hash mismatch: {actual_text_hash}')
    log(f'Text hash verified: {actual_text_hash} (length={len(SEGMENT_0_TEXT)})')

    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f'Wrong runtime Python {sys.version_info.major}.{sys.version_info.minor}; select Colab past runtime 2026.07 / Python 3.12.x, then rerun.')
    import torch
    gpu = torch.cuda.is_available()
    manifest['gpu'] = {'cuda_available': gpu, 'device': torch.cuda.get_device_name(0) if gpu else None, 'torch_version': torch.__version__}
    log(f"Runtime Python={sys.version.split()[0]}; Torch={torch.__version__}; GPU={manifest['gpu']}")
    if not gpu:
        raise RuntimeError('No CUDA GPU found. Select a GPU runtime; CPU inference is not authorized for this pilot.')

    # System and Python dependencies are installed only in this disposable hosted runtime.
    run_command(['apt-get', 'update', '-qq'], 'apt-get update')
    run_command(['apt-get', 'install', '-y', '-qq', 'espeak-ng'], 'install espeak-ng')
    if False:
        run_command([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'kokoro==0.9.4', 'soundfile'], 'install Kokoro')
    else:
        # Fresh Colab runtime, Qwen's documented package pin set resolves inside this runtime.
        run_command([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'qwen-tts==0.1.1'], 'install Qwen3-TTS')
    manifest['package_install_status'] = 'PASS'

    import numpy as np
    import soundfile as sf
    manifest['model_initialization_status'] = 'started'
    if False:
        from kokoro import KPipeline
        log('Initializing Kokoro KPipeline on CUDA...')
        pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M', device='cuda')
        manifest['model_initialization_status'] = 'PASS'
        manifest['voice'] = 'af_heart'
        manifest['language'] = 'American English (lang_code=a)'
        chunks = []
        log('Running actual Kokoro inference...')
        for _, _, audio in pipeline(SEGMENT_0_TEXT, voice='af_heart'):
            if audio is not None:
                chunks.append(np.asarray(audio, dtype=np.float32).reshape(-1))
        if not chunks:
            raise RuntimeError('KPipeline returned no audio chunks.')
        audio = np.concatenate(chunks)
        sample_rate = 24000
    else:
        # Probe actual public repository access before requesting its weights.
        from huggingface_hub import HfApi
        from qwen_tts import Qwen3TTSModel
        model_id = 'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice'
        log(f'Checking Hugging Face model metadata for {model_id} ...')
        info = HfApi().model_info(model_id)
        manifest['model_revision'] = getattr(info, 'sha', None)
        log(f'Model repository lookup PASS; revision={manifest['model_revision']}')
        use_bf16 = bool(torch.cuda.is_bf16_supported())
        dtype = torch.bfloat16 if use_bf16 else torch.float16
        manifest['dtype'] = str(dtype)
        log(f'Initializing Qwen3TTSModel with dtype={dtype}; FlashAttention is intentionally not required.')
        model = Qwen3TTSModel.from_pretrained(model_id, device_map='cuda:0', dtype=dtype)
        manifest['model_initialization_status'] = 'PASS'
        speakers = model.get_supported_speakers() or []
        languages = model.get_supported_languages() or []
        manifest['supported_speakers'] = speakers
        manifest['supported_languages'] = languages
        if 'Ryan' not in speakers:
            raise RuntimeError(f'Expected documented English speaker Ryan not found in supported speakers: {speakers}')
        if 'English' not in languages:
            raise RuntimeError(f'English not listed in supported languages: {languages}')
        manifest['voice'] = 'Ryan'
        manifest['language'] = 'English'
        log('Running actual Qwen3-TTS CustomVoice inference...')
        wavs, sample_rate = model.generate_custom_voice(text=SEGMENT_0_TEXT, language='English', speaker='Ryan')
        if not wavs:
            raise RuntimeError('Qwen generate_custom_voice returned no audio arrays.')
        audio = np.asarray(wavs[0], dtype=np.float32).reshape(-1)

    manifest['inference_status'] = 'PASS'
    if audio.size == 0:
        raise RuntimeError('Inference returned an empty audio array.')
    if not np.isfinite(audio).all():
        raise RuntimeError('Audio contains NaN or infinite values.')
    sf.write(AUDIO_PATH, audio, int(sample_rate))
    check, read_sr = sf.read(AUDIO_PATH, dtype='float32', always_2d=True)
    if check.size == 0 or read_sr != int(sample_rate):
        raise RuntimeError('Saved WAV failed read-back/sample-rate validation.')
    duration_seconds = float(check.shape[0] / read_sr)
    peak_abs = float(np.max(np.abs(check)))
    rms = float(np.sqrt(np.mean(np.square(check))))
    if duration_seconds <= 0 or peak_abs < 1e-5:
        raise RuntimeError(f'Audio exists but duration/signal-level sanity check failed: duration={duration_seconds}, peak={peak_abs}.')
    manifest.update({
      'technical_validation_status': 'PASS',
      'audio_sha256': hashlib.sha256(AUDIO_PATH.read_bytes()).hexdigest(),
      'audio_bytes': AUDIO_PATH.stat().st_size,
      'sample_rate': int(read_sr),
      'channels': int(check.shape[1]),
      'duration_seconds': duration_seconds,
      'peak_abs': peak_abs,
      'rms': rms,
      'audio_quality_status': 'PENDING_HUMAN_LISTENING',
      'overall_status': 'PASS_TECHNICAL_REVIEW_PENDING',
    })
    log(f"WAV written and validated: {AUDIO_PATH}; sr={read_sr}; duration={manifest['duration_seconds']:.3f}s; channels={check.shape[1]}; sha256={manifest['audio_sha256']}")
except Exception as exc:
    manifest['overall_status'] = 'PARTIAL_OR_FAILED'
    manifest['error'] = repr(exc)
    manifest['traceback'] = traceback.format_exc()
    log('ERROR: ' + repr(exc))
    log(manifest['traceback'])
finally:
    write_diagnostics()
    print('\nMANIFEST:')
    print(json.dumps(manifest, indent=2))
    try:
        from google.colab import files
        files.download(str(ZIP_PATH))
    except Exception as download_exc:
        print(f'Automatic download not available: {download_exc}; use the Colab file browser to retrieve {ZIP_PATH}')

## Interpreting the outcome

- `PASS_TECHNICAL_REVIEW_PENDING` means real inference ran and the WAV passed basic file/signal checks. **It does not mean the voice has been accepted.** Listen to the WAV before making an editorial decision.
- `PARTIAL_OR_FAILED` means inspect `execution.log`, `manifest.json`, and the full traceback in the manifest. Do not retry blindly; use the actual error to choose the next correction.
- The ZIP downloaded by the final cell contains the audio (if produced), manifest and execution log. Keep it as evidence; do not import it into production yet.